In [1]:
import subprocess
import sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "pyreadstat"],
    check=True,
)


CompletedProcess(args=['D:\\Works\\Persona agent\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'pandas', 'pyreadstat'], returncode=0)

In [6]:
from collections import Counter
import gzip
import hashlib
import json
import math
from pathlib import Path
import re
from types import SimpleNamespace
from typing import Any

import pandas as pd

WORK = Path("gss")
WORK.mkdir(parents=True, exist_ok=True)

INPUT_DTA = WORK / "gss7224_r3a.dta"
SCHEMA_PATH = Path("schema/dimensions.json")
OUTPUT_PATH = WORK / "gss_personas_1290.jsonl.gz"
MANIFEST_PATH = WORK / "gss_personas_1290.manifest.json"

# 100 để smoke test; đổi thành 0 để xử lý toàn bộ 75.699 respondent.
ROW_LIMIT = 100
CHUNK_SIZE = 2000

print("Input:", INPUT_DTA)
print("Schema:", SCHEMA_PATH)
print("Output:", OUTPUT_PATH)

Input: gss\gss7224_r3a.dta
Schema: schema\dimensions.json
Output: gss\gss_personas_1290.jsonl.gz


In [7]:
GSS_USED_COLUMNS = [
    "age",
    "sex",
    "race",
    "hispanic",
    "region",
    "born",
    "marital",
    "childs",
    "degree",
    "educ",
    "wrkstat",
    "class",
    "polviews",
    "relig",
    "reliten",
    "health",
    "trust",
    "fefam",
]
SAMPLE_SIZE = 10

if not INPUT_DTA.is_file():
    raise FileNotFoundError(f"Không tìm thấy GSS: {INPUT_DTA}")

sample_reader = pd.read_stata(
    INPUT_DTA,
    columns=GSS_USED_COLUMNS,
    convert_categoricals=True,
    iterator=True,
)
gss_sample = sample_reader.read(SAMPLE_SIZE)

print("Kích thước sample:", gss_sample.shape)
print("Các cột pipeline sử dụng:", gss_sample.columns.tolist())
display(gss_sample)

Kích thước sample: (10, 18)
Các cột pipeline sử dụng: ['age', 'sex', 'race', 'hispanic', 'region', 'born', 'marital', 'childs', 'degree', 'educ', 'wrkstat', 'class', 'polviews', 'relig', 'reliten', 'health', 'trust', 'fefam']


C:\Users\Duong\AppData\Local\Temp\ipykernel_31612\3452293330.py:32: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  gss_sample = sample_reader.read(SAMPLE_SIZE)


,age,sex,race,hispanic,region,born,marital,childs,degree,educ,wrkstat,class,polviews,relig,reliten,health,trust,fefam
0,23,female,white,NaN,midwest,NaN,never married,0,bachelor's,4 years of college,working full time,middle class,NaN,jewish,NaN,good,depends,NaN
1,70,male,white,NaN,midwest,NaN,married,5,less than high school,10th grade,retired,middle class,NaN,catholic,NaN,fair,most people can be trusted,NaN
2,48,female,white,NaN,midwest,NaN,married,4,high school,12th grade,working part time,working class,NaN,protestant,NaN,excellent,can't be too careful,NaN
3,27,female,white,NaN,midwest,NaN,married,0,bachelor's,5 years of college,working full time,middle class,NaN,other,NaN,good,can't be too careful,NaN
4,61,female,white,NaN,midwest,NaN,married,2,high school,12th grade,keeping house,working class,NaN,protestant,NaN,good,can't be too careful,NaN
5,26,male,white,NaN,midwest,NaN,never married,0,high school,2 years of college,working full time,middle class,NaN,protestant,NaN,good,most people can be trusted,NaN
6,28,male,white,NaN,midwest,NaN,divorced,2,high school,1 year of college,working full time,middle class,NaN,catholic,NaN,excellent,can't be too careful,NaN
7,27,male,white,NaN,midwest,NaN,never married,0,bachelor's,4 years of college,working full time,working class,NaN,jewish,NaN,good,most people can be trusted,NaN
8,21,female,black,NaN,south,NaN,never married,2,high school,12th grade,working part time,working class,NaN,protestant,NaN,excellent,can't be too careful,NaN
9,30,female,black,NaN,south,NaN,married,4,high school,12th grade,working full time,working class,NaN,protestant,NaN,fair,can't be too careful,NaN


In [10]:
def missing(value: Any) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def text(row: Any, key: str) -> str | None:
    value = row.get(key)
    return None if missing(value) else source_key(value)


def source_key(value: Any) -> str:
    """Canonical form used for all decoded Stata labels and numeric codes."""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        number = float(value)
        if math.isfinite(number) and number.is_integer():
            # GSS mappings historically use decimal strings such as "0.0".
            return f"{number:.1f}"
    normalized = str(value).strip().lower()
    normalized = normalized.replace("’", "'").replace("‘", "'")
    normalized = re.sub(r"\s+", " ", normalized)
    return normalized


def age_bracket(row: Any) -> str | None:
    value = row.get("age")
    if missing(value):
        return None
    if "older" in str(value).lower():
        return "85+"
    try:
        age = float(value)
    except (TypeError, ValueError):
        return None
    for low, high, label in (
        (18, 24, "18-24"), (25, 34, "25-34"), (35, 44, "35-44"),
        (45, 54, "45-54"), (55, 64, "55-64"), (65, 74, "65-74"),
        (75, 84, "75-84"),
    ):
        if low <= age <= high:
            return label
    return "85+" if age >= 85 else None


def ethnicity(row: Any) -> str | None:
    hispanic, race = text(row, "hispanic"), text(row, "race")
    if hispanic and hispanic != "not hispanic":
        return "Hispanic / Latino"
    return {"white": "White / European", "black": "Black / African"}.get(race)


def education(row: Any) -> str | None:
    degree, educ = text(row, "degree"), text(row, "educ")
    if degree == "graduate":
        return "Doctorate" if educ and ("7 years" in educ or "8 or more years" in educ) else "Master's"
    if degree == "bachelor's": return "Bachelor's"
    if degree == "associate/junior college": return "Associate's"
    if degree == "high school": return "Secondary"
    if degree == "less than high school":
        match = re.match(r"(\d+)", educ or "")
        return "Primary" if match and int(match.group(1)) <= 8 else "Secondary"
    return None


def religiosity(row: Any) -> str | None:
    religion, intensity = text(row, "relig"), text(row, "reliten")
    if religion == "none" or intensity == "no religion": return "Secular"
    return {"strong": "Devout", "somewhat strong (vol.)": "Observant",
            "not very strong": "Spiritual"}.get(intensity)


def region(row: Any) -> str | None:
    return "North America" if text(row, "region") else None


def seniority(row: Any) -> str | None:
    return {"retired": "Retired", "in school": "Student / intern"}.get(text(row, "wrkstat"))


def life_stage(row: Any) -> str | None:
    return {"in school": "Student", "retired": "Retirement"}.get(text(row, "wrkstat"))


# A target value of None deliberately means "leave unsupported", not a missing rule.
CROSSWALK: dict[str, dict[str, Any]] = {
    "age_bracket": {"compute": age_bracket},
    "gender_identity": {"src": "sex", "map": {"female": "Woman", "male": "Man"}},
    "demo_ethnicity_broad": {"compute": ethnicity},
    "demo_citizenship_status": {"src": "born", "map": {"yes": "Citizen by birth", "no": None}},
    "demo_children_count": {"src": "childs", "map": {
        "0": "None", "0.0": "None", "1": "1 child", "1.0": "1 child",
        "2": "2 children", "2.0": "2 children", "3": "3+ children", "3.0": "3+ children",
        "4": "3+ children", "5": "3+ children", "6": "3+ children", "7": "3+ children",
        "4.0": "3+ children", "5.0": "3+ children", "6.0": "3+ children",
        "7.0": "3+ children", "8 or more": "3+ children"}},
    "demo_marital_status": {"src": "marital", "map": {
        "married": "Married", "never married": "Single", "divorced": "Divorced",
        "widowed": "Widowed", "separated": "Separated"}},
    "highest_education": {"compute": education},
    "socioeconomic_band": {"src": "class", "map": {
        "lower class": "Low income", "working class": "Lower-middle",
        "middle class": "Middle", "upper class": "Upper-middle", "no class": None}},
    "demo_employment_status": {"src": "wrkstat", "map": {
        "working full time": "Full-time", "working part time": "Part-time",
        "retired": "Retired", "keeping house": "Homemaker",
        "unemployed, laid off, looking for work": "Unemployed", "in school": "Student",
        "with a job, but not at work because of temporary illness, vacation, strike": "Full-time",
        "other": None}},
    "seniority": {"compute": seniority},
    "life_stage": {"compute": life_stage},
    "political_lean": {"src": "polviews", "map": {
        "extremely liberal": "Left", "liberal": "Left", "slightly liberal": "Center-left",
        "moderate, middle of the road": "Center", "slightly conservative": "Center-right",
        "conservative": "Right", "extremely conservative": "Right"}},
    "religiosity": {"compute": religiosity},
    "demo_religion_affiliation": {"src": "relig", "map": {
        "protestant": "Christian", "catholic": "Christian", "christian": "Christian",
        "orthodox-christian": "Christian", "jewish": "Jewish", "muslim/islam": "Muslim",
        "buddhism": "Buddhist", "hinduism": "Hindu", "none": "None",
        "native american": "Folk / traditional", "other": None,
        "inter-nondenominational": None, "other eastern religions": None}},
    "health_general_health": {"src": "health", "map": {
        "excellent": "Excellent", "good": "Good", "fair": "Fair", "poor": "Poor"}},
    "trust_level": {"src": "trust", "map": {
        "most people can be trusted": "Trusting", "can't be too careful": "Skeptical",
        "depends": "Verifying"}},
    "region": {"compute": region},
    "att_traditional_gender_roles": {"src": "fefam", "map": {
        "strongly agree": "Enthusiast", "agree": "Positive",
        "disagree": "Skeptical", "strongly disagree": "Opposed"}},
}


def load_schema(path: Path) -> tuple[list[str], dict[str, set[Any]]]:
    dimensions = json.loads(path.read_text(encoding="utf-8"))["dimensions"]
    order = [item["id"] for item in dimensions]
    if len(order) != 1290 or len(set(order)) != len(order):
        raise ValueError(f"Expected 1,290 unique schema dimensions, found {len(order)}")
    return order, {item["id"]: set(item.get("values") or []) for item in dimensions}


def apply_crosswalk(row: Any, allowed: dict[str, set[Any]]) -> tuple[dict[str, Any], dict[str, Any]]:
    observed, unmapped = {}, {}
    for dimension_id, rule in CROSSWALK.items():
        if dimension_id not in allowed:
            raise KeyError(f"Crosswalk targets unknown dimension: {dimension_id}")
        if "compute" in rule:
            target = rule["compute"](row)
        else:
            raw = row.get(rule["src"])
            if missing(raw):
                continue
            key = source_key(raw)
            if key not in rule["map"]:
                unmapped[dimension_id] = str(raw)
                continue
            target = rule["map"][key]
        if target is None:
            continue
        if target not in allowed[dimension_id]:
            raise ValueError(f"{dimension_id}: off-schema value {target!r}")
        observed[dimension_id] = target
    return observed, unmapped


def full_fields(order: list[str], observed: dict[str, Any]) -> list[dict[str, Any]]:
    return [
        {"field_id": dimension_id, "value": observed.get(dimension_id),
         "confidence": 1.0 if dimension_id in observed else 0.0,
         "evidence": "Direct GSS survey response" if dimension_id in observed else "",
         "description": "", "assignment_type": "direct" if dimension_id in observed else "unsupported"}
        for dimension_id in order
    ]


def audit_crosswalk_row(row: Any, allowed: dict[str, set[Any]]) -> list[dict[str, Any]]:
    """Explain every GSS crosswalk decision for one respondent."""
    observed, unmapped = apply_crosswalk(row, allowed)
    report = []
    for dimension_id, rule in CROSSWALK.items():
        if "compute" in rule:
            value = rule["compute"](row)
            report.append({
                "dimension_id": dimension_id,
                "source_column": "compute",
                "raw_value": None,
                "normalized_value": None,
                "result": value,
                "status": "mapped" if value is not None else "unsupported",
            })
            continue
        source_column = rule["src"]
        raw = row.get(source_column)
        key = None if missing(raw) else source_key(raw)
        if missing(raw):
            status, result = "missing", None
        elif key not in rule["map"]:
            status, result = "unmapped", None
        elif rule["map"][key] is None:
            status, result = "deliberately_null", None
        else:
            status, result = "mapped", rule["map"][key]
        report.append({
            "dimension_id": dimension_id,
            "source_column": source_column,
            "raw_value": None if missing(raw) else str(raw),
            "normalized_value": key,
            "result": result,
            "status": status,
        })
    return report


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def convert(args: SimpleNamespace) -> dict[str, Any]:
    order, allowed = load_schema(args.schema)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    unmapped_counts: Counter[str] = Counter()
    grounded_counts: Counter[int] = Counter()
    rows_written = 0

    # Load only 18 small crosswalk columns at once. This is modest in memory for
    # 75,699 respondents and lets pandas see each complete value-label table.
    # StataReader chunk conversion can otherwise produce inconsistent category
    # sets and repeated CategoricalConversionWarning messages.
    source = pd.read_stata(
        args.input,
        # Read only the 18 crosswalk inputs. The cumulative GSS file contains
        # unrelated columns (for example paisco08) whose Stata value-label
        # tables have duplicate labels; asking pandas to categorical-convert
        # those unused columns raises "Categorical categories must be unique".
        columns=GSS_USED_COLUMNS,
        convert_categoricals=True,
    )
    with gzip.open(args.output, "wt", encoding="utf-8") as output:
        for chunk_start in range(0, len(source), args.chunk_size):
            frame = source.iloc[chunk_start : chunk_start + args.chunk_size]
            for source_index, row in frame.iterrows():
                observed, unmapped = apply_crosswalk(row, allowed)
                for dimension_id, raw in unmapped.items():
                    unmapped_counts[f"{dimension_id}={raw}"] += 1
                fields = full_fields(order, observed)
                if len(fields) != 1290:
                    raise AssertionError("Output field count is not 1,290")
                user_id = f"gss-{rows_written:06d}"
                output.write(json.dumps({
                    "user_id": user_id, "source": "gss_cumulative_1972_2024",
                    "source_row_index": str(source_index), "fields": fields,
                    "observed": observed,
                }, ensure_ascii=False) + "\n")
                grounded_counts[len(observed)] += 1
                rows_written += 1
                if args.limit and rows_written >= args.limit:
                    break
            print(f"Converted {rows_written:,} respondents", flush=True)
            if args.limit and rows_written >= args.limit:
                break

    manifest = {
        "source": str(args.input), "source_sha256": file_sha256(args.input),
        "schema": str(args.schema), "schema_sha256": file_sha256(args.schema),
        "output": str(args.output), "output_sha256": file_sha256(args.output),
        "rows": rows_written, "fields_per_row": 1290,
        "crosswalk_dimension_count": len(CROSSWALK),
        "grounded_dimension_histogram": dict(sorted(grounded_counts.items())),
        "top_unmapped_values": unmapped_counts.most_common(100),
    }
    manifest_path = args.manifest or args.output.with_suffix(args.output.suffix + ".manifest.json")
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return manifest


def validate(path: Path, schema: Path, expected_rows: int | None = None) -> dict[str, int]:
    order, allowed = load_schema(schema)
    rows = errors = 0
    user_ids: set[str] = set()
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            row = json.loads(line); rows += 1
            fields = row.get("fields") or []
            if len(fields) != len(order) or [field.get("field_id") for field in fields] != order:
                errors += 1; continue
            if row.get("user_id") in user_ids: errors += 1
            user_ids.add(row.get("user_id"))
            for field in fields:
                value = field.get("value")
                if value is not None and value not in allowed[field["field_id"]]: errors += 1
                expected_type = "direct" if value is not None else "unsupported"
                if field.get("assignment_type") != expected_type: errors += 1
    if expected_rows is not None and rows != expected_rows: errors += 1
    report = {"rows": rows, "unique_users": len(user_ids), "errors": errors}
    if errors:
        raise ValueError(f"Validation failed: {report}")
    return report


In [ ]:
if not INPUT_DTA.is_file():
    raise FileNotFoundError(f"Không tìm thấy GSS: {INPUT_DTA}")
if not SCHEMA_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy schema: {SCHEMA_PATH}")

config = SimpleNamespace(
    input=INPUT_DTA,
    schema=SCHEMA_PATH,
    output=OUTPUT_PATH,
    manifest=MANIFEST_PATH,
    chunk_size=CHUNK_SIZE,
    limit=ROW_LIMIT,
)
manifest = convert(config)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

In [11]:
validation_report = validate(
    OUTPUT_PATH,
    SCHEMA_PATH,
    expected_rows=manifest["rows"],
)
print("Validation:", validation_report)

Validation: {'rows': 100, 'unique_users': 100, 'errors': 0}


In [14]:
with gzip.open(OUTPUT_PATH, "rt", encoding="utf-8") as handle:
    first_persona = json.loads(handle.readline())

supported = [field for field in first_persona["fields"] if field["value"] is not None]
print("User:", first_persona["user_id"])
print("Tổng fields:", len(first_persona["fields"]))
print("Observed fields:", len(supported))
print(json.dumps(supported, ensure_ascii=False, indent=2))

User: gss-000000
Tổng fields: 1290
Observed fields: 12
[
  {
    "field_id": "age_bracket",
    "value": "18-24",
    "confidence": 1.0,
    "evidence": "Direct GSS survey response",
    "description": "",
    "assignment_type": "direct"
  },
  {
    "field_id": "region",
    "value": "North America",
    "confidence": 1.0,
    "evidence": "Direct GSS survey response",
    "description": "",
    "assignment_type": "direct"
  },
  {
    "field_id": "gender_identity",
    "value": "Woman",
    "confidence": 1.0,
    "evidence": "Direct GSS survey response",
    "description": "",
    "assignment_type": "direct"
  },
  {
    "field_id": "socioeconomic_band",
    "value": "Middle",
    "confidence": 1.0,
    "evidence": "Direct GSS survey response",
    "description": "",
    "assignment_type": "direct"
  },
  {
    "field_id": "highest_education",
    "value": "Bachelor's",
    "confidence": 1.0,
    "evidence": "Direct GSS survey response",
    "description": "",
    "assignment_type": "